# 🌲 MÓDULO 3: Entrenamiento de Modelos
## Árbol de Decisión vs Random Forest

**Objetivo:** Entrenar ambos modelos y comparar su rendimiento.

---

## 3.1 Configuración y Preparación de Datos

In [1]:
import sys
import time

EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    print("📍 Ejecutando en Google Colab")
    !pip install pyspark -q
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_DATOS = '/content/drive/MyDrive/ML_Vuelos/data/'  # Ajustar según tu Drive
else:
    print("🐳 Ejecutando en Docker")
    RUTA_DATOS = '../data/'

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Vuelos_Modulo3") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

🐳 Ejecutando en Docker


In [2]:
# Cargar y preparar datos (repetimos del Módulo 2)
from pyspark.ml.feature import StringIndexer, VectorAssembler

df = spark.read.csv(RUTA_DATOS + "flights_2015_full.csv", header=True, inferSchema=True)

# Indexar categóricas
for col_name in ["AIRLINE", "ORIGIN", "DEST"]:
    indexer = StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx", handleInvalid="keep")
    df = indexer.fit(df).transform(df)

# Crear vector de features
feature_cols = ["MONTH", "DAY_OF_WEEK", "DEP_HOUR", "DISTANCE", "AIRLINE_idx", "ORIGIN_idx", "DEST_idx"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_ml = assembler.transform(df).select("features", "DEP_DEL15")

# Dividir
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f"✅ Datos preparados: {train_df.count():,} train / {test_df.count():,} test")

✅ Datos preparados: 4,266,480 train / 1,066,434 test


## 3.2 Entrenar Árbol de Decisión 🌳

In [3]:
from pyspark.ml.classification import DecisionTreeClassifier
import time

print("🌳 Entrenando Árbol de Decisión...")
inicio = time.time()

dt = DecisionTreeClassifier(
    labelCol="DEP_DEL15",
    featuresCol="features",
    maxDepth=5,          # Profundidad máxima
    minInstancesPerNode=100,  # Mínimo de ejemplos por nodo
    maxBins=500          # Ajustado para acomodar características categóricas con muchos valores únicos
)

modelo_arbol = dt.fit(train_df)
tiempo_arbol = time.time() - inicio

print(f"✅ Árbol entrenado en {tiempo_arbol:.2f} segundos")
print(f"   Profundidad: {modelo_arbol.depth}")
print(f"   Nodos: {modelo_arbol.numNodes}")

🌳 Entrenando Árbol de Decisión...


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_com

Py4JError: An error occurred while calling o197.fit

In [4]:
# Feature Importance del Árbol
print("\n📊 IMPORTANCIA DE VARIABLES (Árbol)")
print("="*40)
importancias = modelo_arbol.featureImportances.toArray()
for nombre, imp in sorted(zip(feature_cols, importancias), key=lambda x: -x[1]):
    barra = "█" * int(imp * 50)
    print(f"{nombre:15} {imp:.4f} {barra}")


📊 IMPORTANCIA DE VARIABLES (Árbol)


NameError: name 'modelo_arbol' is not defined

In [5]:
# Ver estructura del árbol
print("\n🌳 ESTRUCTURA DEL ÁRBOL (primeras líneas):")
print(modelo_arbol.toDebugString[:2000])


🌳 ESTRUCTURA DEL ÁRBOL (primeras líneas):


NameError: name 'modelo_arbol' is not defined

## 3.3 Entrenar Random Forest 🌲🌲🌲

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

print("🌲 Entrenando Random Forest (100 árboles)...")
inicio = time.time()

rf = RandomForestClassifier(
    labelCol="DEP_DEL15",
    featuresCol="features",
    numTrees=100,        # Número de árboles
    maxDepth=5,          # Profundidad máxima por árbol
    seed=42,
    maxBins=500          # Ajustado para acomodar características categóricas con muchos valores únicos
)

modelo_bosque = rf.fit(train_df)
tiempo_bosque = time.time() - inicio

print(f"✅ Bosque entrenado en {tiempo_bosque:.2f} segundos")
print(f"   Árboles: {modelo_bosque.numTrees}")

In [7]:
# Feature Importance del Bosque
print("\n📊 IMPORTANCIA DE VARIABLES (Random Forest)")
print("="*40)
importancias_rf = modelo_bosque.featureImportances.toArray()
for nombre, imp in sorted(zip(feature_cols, importancias_rf), key=lambda x: -x[1]):
    barra = "█" * int(imp * 50)
    print(f"{nombre:15} {imp:.4f} {barra}")


📊 IMPORTANCIA DE VARIABLES (Random Forest)


NameError: name 'modelo_bosque' is not defined

## 3.4 Comparación de Tiempos

In [6]:
print("\n⏱️ COMPARACIÓN DE TIEMPOS DE ENTRENAMIENTO")
print("="*45)
print(f"{'Modelo':<25} {'Tiempo':>10} {'Factor':>8}")
print("-"*45)
print(f"{'Árbol de Decisión':<25} {tiempo_arbol:>8.2f}s {1.0:>7.1f}x")
print(f"{'Random Forest (100)':<25} {tiempo_bosque:>8.2f}s {tiempo_bosque/tiempo_arbol:>7.1f}x")
print("="*45)


⏱️ COMPARACIÓN DE TIEMPOS DE ENTRENAMIENTO
Modelo                        Tiempo   Factor
---------------------------------------------


NameError: name 'tiempo_arbol' is not defined

---
## ✅ CHECKPOINT MÓDULO 3

Ahora tienes:

| Modelo | Variable | Tiempo |
|--------|----------|--------|
| Árbol de Decisión | `modelo_arbol` | Rápido |
| Random Forest | `modelo_bosque` | Más lento |

**Siguiente:** → `Modulo4_Evaluacion.ipynb`